# NBT small-set preprocessing

This notebook uses the reusable preprocessing pipeline in `src/nbt_pipeline/preprocessing/`.


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

import matplotlib.pyplot as plt
import pandas as pd

from nbt_pipeline.preprocessing import (
    blank_like_summary,
    build_preprocessed_dataset,
    column_overview,
    load_nbt_smallset,
    missing_summary,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.2f}".format)

nbt_smallset_df = load_nbt_smallset()
preprocessed_df = build_preprocessed_dataset()
nbt_smallset_df.head()


,TheatreRoom,admission_type,SessionIDdesc,intended_management,actual_proc_1_procedure_code,ProcedureDescription,theatre_notes,ExpectedDurationMins,into_theatre,anaesthetic_start_time,incision,closure,out_of_theatre,operation_end_time,recovery_time,operation_length_mins,sex_national_code,age_at_operation,ASAScore,PriorityLevelCode,anaesthetic_desc,listing_cons_code,theat_surg_1_national_code,theat_anae_1_national_code
0,PLASTIC MINOR 2,11,TH 26 AM P Consultant (Plastic Surgery),2.00,S069,Unspecified other excision of lesion of skin,60mins/day case/LA/ 5mm WLE R abdo scar )(in-situ MM) ....,60.00,00:20:00,00:20:00,00:28:00,00:32:00,09:43:00,00:39:00,00:43:00,19.00,2.00,68.00,1.00,P2,LA (Surgeon),A000012,C4090155,NaN
1,PARKVIEW TH02,11,TH2 (2SD) A Harris (Urology),2.00,M653,Endoscopic resection of prostate NEC,TURP DC GA P3 Xray no Comments = Retention doing ISC....,120.00,00:15:00,00:15:00,00:39:00,00:04:00,NaN,00:47:00,NaN,92.00,1.00,72.00,2.00,P3,LA (Surgeon),C7016369,C7016369,C7042030
2,BRUNEL TH 11,11,TH11(2SD) Mr A Kiszely (Trauma and Orthopaedics),2.00,Z724,Carpal bone NEC,Right Trapeziectomy +/- LRTI 90 mins....,120.00,00:58:00,00:58:00,00:23:00,00:50:00,15:04:00,00:58:00,NaN,60.00,2.00,70.00,2.00,P4,GA,C7136326,C7136326,7038820
3,PLASTIC MINOR 2,11,TH 26 PM P Consultant (Plastic Surgery),2.00,S069,Unspecified other excision of lesion of skin,60mnins/day case/LA/ excision 4mm margin dorsum left hand direct closure (MM? Adnexal tumour?) c+c dorsum left hand...,60.00,00:12:00,00:12:00,00:22:00,00:30:00,14:47:00,00:40:00,00:47:00,28.00,2.00,85.00,2.00,NaN,LA (Surgeon),A000012,C7040583,NaN
4,BRUNEL TH 15,11,TH15 (3SD) H GILBERT (Urology),2.00,M455,Diagnostic endoscopic examination of bladder using rigid cystoscope,Rigid cystoscopy under GA +/- Biopsy DC GA....,48.00,00:51:00,00:51:00,00:06:00,00:17:00,10:20:00,00:18:00,NaN,27.00,2.00,74.00,3.00,P2,GA,C3015366,C3398948,C3492598


## 1. Load the data

In [4]:
data_path = Path("../data/NBT-SmallSet.xlsx")

nbt_smallset_df = pd.read_excel(data_path, sheet_name="Sheet1")

In [5]:
pd.concat([nbt_smallset_df.head(3), nbt_smallset_df.tail(3)])

,TheatreRoom,admission_type,SessionIDdesc,intended_management,actual_proc_1_procedure_code,ProcedureDescription,theatre_notes,ExpectedDurationMins,into_theatre,anaesthetic_start_time,incision,closure,out_of_theatre,operation_end_time,recovery_time,operation_length_mins,sex_national_code,age_at_operation,ASAScore,PriorityLevelCode,anaesthetic_desc,listing_cons_code,theat_surg_1_national_code,theat_anae_1_national_code
0,PLASTIC MINOR 2,11,TH 26 AM P Consultant (Plastic Surgery),2.00,S069,Unspecified other excision of lesion of skin,60mins/day case/LA/ 5mm WLE R abdo scar )(in-situ MM) ....,60.00,00:20:00,00:20:00,00:28:00,00:32:00,09:43:00,00:39:00,00:43:00,19.00,2.00,68.00,1.00,P2,LA (Surgeon),A000012,C4090155,NaN
1,PARKVIEW TH02,11,TH2 (2SD) A Harris (Urology),2.00,M653,Endoscopic resection of prostate NEC,TURP DC GA P3 Xray no Comments = Retention doing ISC....,120.00,00:15:00,00:15:00,00:39:00,00:04:00,NaN,00:47:00,NaN,92.00,1.00,72.00,2.00,P3,LA (Surgeon),C7016369,C7016369,C7042030
2,BRUNEL TH 11,11,TH11(2SD) Mr A Kiszely (Trauma and Orthopaedics),2.00,Z724,Carpal bone NEC,Right Trapeziectomy +/- LRTI 90 mins....,120.00,00:58:00,00:58:00,00:23:00,00:50:00,15:04:00,00:58:00,NaN,60.00,2.00,70.00,2.00,P4,GA,C7136326,C7136326,7038820
14915,PARKVIEW TH02,11,"TH2 (2SD) S,Bolomytis (Urology)",2.00,M455,Diagnostic endoscopic examination of bladder using rigid cystoscope,m45.5 Cystoscopy and bladder biopsies DC....,60.00,00:48:00,00:48:00,00:45:00,00:54:00,NaN,00:00:00,NaN,72.00,1.00,76.00,3.00,P2,GA,C7477113,C7477113,C3467499
14916,PLASTIC MINOR 1,11,TH25 AM P Consultant (Plastic Surgery),2.00,S069,Unspecified other excision of lesion of skin,90mins/day case/LA/ bcc right nasal dorsum side wall excise and local flap/FTSG staff grade PSOP P2 within 2-...,90.00,00:17:00,00:17:00,00:36:00,00:01:00,10:28:00,00:23:00,00:28:00,66.00,1.00,63.00,2.00,P2,LA (Surgeon),A000012,C6156511,NaN
14917,IR LAB 4,12,IR4 Morning / R Consultant (Radiology),2.00,M131,Percutaneous needle biopsy of lesion of kidney,33mm right lower pole tumour being worked up for cryoablation. For histological confirmation....,45.00,00:00:00,00:00:00,00:23:00,00:32:00,10:41:00,00:33:00,NaN,33.00,1.00,72.00,NaN,P2,LA (Surgeon),A000013,C4717339,NaN


## 2. Data dictionary

This section briefly explains the main columns before preprocessing. Some meanings come from NHS Data Dictionary / NHS England sources, while local operational fields are interpreted from the column names and values in this dataset.

| Column | Brief meaning | Source / note |
|---|---|---|
| `TheatreRoom` | Operating theatre room where the case was planned or performed. | Dataset context; related to operating theatre activity. |
| `admission_type` | Code describing the method/type of admission, for example elective waiting list, booked, planned, or emergency admission. | NHS Data Dictionary: Admission Method. |
| `SessionIDdesc` | Text description of the theatre session, often including theatre number, session timing, consultant, or specialty. | Dataset context; related to NHS operating theatre session concepts. |
| `intended_management` | Planned hospital bed use, for example whether the patient is expected to stay overnight or be treated as a day case. | NHS Data Dictionary: Intended Management. |
| `actual_proc_1_procedure_code` | OPCS-4 code for the main procedure/intervention. | NHS England OPCS-4 classification. |
| `ProcedureDescription` | Text description linked to the main OPCS procedure code. | NHS England OPCS-4 classification. |
| `theatre_notes` | Free-text notes entered for the theatre case, sometimes containing procedure details, duration, anaesthetic notes, or special requirements. | Dataset context/local operational notes. |
| `ExpectedDurationMins` | Planned or expected case duration in minutes. | Dataset context. |
| `into_theatre`, `anaesthetic_start_time`, `incision`, `closure`, `out_of_theatre`, `operation_end_time`, `recovery_time` | Theatre pathway timestamps showing when key stages happened. | Dataset context; used to calculate timings and delays. |
| `operation_length_mins` | Recorded operation duration in minutes. | Dataset context. |
| `sex_national_code` | National code for patient sex/gender category, commonly 1 = male and 2 = female in NHS dictionary coding. | NHS Data Dictionary: Person Stated Gender Code / Person Gender Code. |
| `age_at_operation` | Patient age at the time of operation. | Dataset context. |
| `ASAScore` | ASA physical status score, showing patient fitness/comorbidity risk before anaesthesia. | NHS Data Dictionary: ASA Physical Status Classification System Code. |
| `PriorityLevelCode` | Clinical priority code, such as P2, P3, or P4, indicating urgency/timeframe for elective care. | NHS England elective access policy. |
| `anaesthetic_desc` | Description of anaesthetic type, for example general anaesthetic or local anaesthetic. | Dataset context. |
| `listing_cons_code` | Code for the listing consultant. | Dataset context/local staff code. |
| `theat_surg_1_national_code` | Code for the first theatre surgeon. | Dataset context/local or national staff code. |
| `theat_anae_1_national_code` | Code for the first theatre anaesthetist. | Dataset context/local or national staff code. |


## 3. Basic structure

In [6]:
rows, columns = nbt_smallset_df.shape
print(f"Rows: {rows:,}")
print(f"Columns: {columns:,}")

Rows: 14,918
Columns: 24


In [7]:
column_overview = pd.DataFrame({
    "column": nbt_smallset_df.columns,
    "dtype": nbt_smallset_df.dtypes.astype(str).values,
    "non_null": nbt_smallset_df.notna().sum().values,
    "missing": nbt_smallset_df.isna().sum().values,
    "missing_pct": (nbt_smallset_df.isna().mean() * 100).round(2).values,
    "unique_values": nbt_smallset_df.nunique(dropna=True).values,
})

column_overview.sort_values("missing_pct", ascending=False)

,column,dtype,non_null,missing,missing_pct,unique_values
14,recovery_time,object,1768,13150,88.15,60
19,PriorityLevelCode,str,6355,8563,57.40,4
23,theat_anae_1_national_code,object,9512,5406,36.24,323
18,ASAScore,float64,13664,1254,8.41,7
20,anaesthetic_desc,str,14174,744,4.99,7
12,out_of_theatre,object,14321,597,4.00,1193
7,ExpectedDurationMins,float64,14704,214,1.43,189
22,theat_surg_1_national_code,object,14803,115,0.77,390
4,actual_proc_1_procedure_code,str,14828,90,0.60,1038
5,ProcedureDescription,str,14828,90,0.60,1038


### How to read this table

This table gives a quick health check for every column in the dataset.

- `non_null`: how many rows have a real value in that column.
- `missing`: how many rows are empty in that column.
- `missing_pct`: the percentage of rows that are empty. A high value means the column may be less useful or needs cleaning.
- `unique_values`: how many different values appear in that column. This helps us understand whether the column is categorical, repeated, or very detailed.

For example, if a column has high `missing_pct`, we should be careful using it. If it has only a few `unique_values`, it may be useful for grouping or modelling as a category.
\n

In [25]:
nbt_smallset_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14918 entries, 0 to 14917
Data columns (total 24 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   TheatreRoom                   14918 non-null  str    
 1   admission_type                14916 non-null  object 
 2   SessionIDdesc                 14918 non-null  str    
 3   intended_management           14913 non-null  float64
 4   actual_proc_1_procedure_code  14828 non-null  str    
 5   ProcedureDescription          14828 non-null  str    
 6   theatre_notes                 14842 non-null  str    
 7   ExpectedDurationMins          14704 non-null  float64
 8   into_theatre                  14918 non-null  object 
 9   anaesthetic_start_time        14918 non-null  object 
 10  incision                      14862 non-null  object 
 11  closure                       14845 non-null  object 
 12  out_of_theatre                14321 non-null  object 
 13  operation_en

## 3. Data quality checks

In [26]:
missing_summary = (
    nbt_smallset_df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(nbt_smallset_df) * 100).round(2)
missing_summary.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
recovery_time,13150,88.15
PriorityLevelCode,8563,57.40
theat_anae_1_national_code,5406,36.24
ASAScore,1254,8.41
anaesthetic_desc,744,4.99
out_of_theatre,597,4.00
ExpectedDurationMins,214,1.43
theat_surg_1_national_code,115,0.77
actual_proc_1_procedure_code,90,0.60
ProcedureDescription,90,0.60


In [ ]:
duplicate_count = nbt_smallset_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count:,}")

nbt_smallset_df[nbt_smallset_df.duplicated(keep=False)].sort_values(nbt_smallset_df.columns.tolist()).head(20)

In [ ]:
text_columns = nbt_smallset_df.select_dtypes(include="object").columns

blank_like_summary = []
for column in text_columns:
    cleaned = nbt_smallset_df[column].astype("string").str.strip()
    blank_like_summary.append({
        "column": column,
        "empty_strings": (cleaned == "").sum(),
        "single_dots": (cleaned == ".").sum(),
        "unique_values": cleaned.nunique(dropna=True),
    })

pd.DataFrame(blank_like_summary).sort_values(["single_dots", "empty_strings"], ascending=False)

## 4. Numeric variable inspection

In [27]:
numeric_columns = [
    "ExpectedDurationMins",
    "operation_length_mins",
    "age_at_operation",
    "ASAScore",
    "intended_management",
    "sex_national_code",
]

nbt_smallset_df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
ExpectedDurationMins,14704.00,98.28,88.53,0.00,60.00,60.00,120.00,1415.00
operation_length_mins,14845.00,97.89,92.50,0.00,39.00,70.00,124.00,1017.00
age_at_operation,14872.00,57.07,19.59,16.00,40.00,59.00,73.00,107.00
ASAScore,13664.00,2.25,0.83,0.00,2.00,2.00,3.00,6.00
intended_management,14913.00,1.56,0.85,1.00,1.00,1.00,2.00,8.00
sex_national_code,14872.00,1.53,0.50,1.00,1.00,2.00,2.00,2.00


In [28]:
numeric_quality_checks = pd.DataFrame({
    "zero_expected_duration": [(nbt_smallset_df["ExpectedDurationMins"] == 0).sum()],
    "zero_operation_length": [(nbt_smallset_df["operation_length_mins"] == 0).sum()],
    "age_under_18": [(nbt_smallset_df["age_at_operation"] < 18).sum()],
    "age_over_100": [(nbt_smallset_df["age_at_operation"] > 100).sum()],
    "asa_score_zero": [(nbt_smallset_df["ASAScore"] == 0).sum()],
    "asa_score_over_5": [(nbt_smallset_df["ASAScore"] > 5).sum()],
})

numeric_quality_checks.T.rename(columns={0: "row_count"})

,row_count
zero_expected_duration,62
zero_operation_length,9
age_under_18,103
age_over_100,2
asa_score_zero,1
asa_score_over_5,21


In [29]:
duration_df = nbt_smallset_df.copy()
duration_df["duration_error_mins"] = duration_df["operation_length_mins"] - duration_df["ExpectedDurationMins"]

duration_df[["ExpectedDurationMins", "operation_length_mins", "duration_error_mins"]].describe().T

,count,mean,std,min,25%,50%,75%,max
ExpectedDurationMins,14704.00,98.28,88.53,0.00,60.00,60.00,120.00,1415.00
operation_length_mins,14845.00,97.89,92.50,0.00,39.00,70.00,124.00,1017.00
duration_error_mins,14632.00,-0.15,69.13,-1399.00,-32.00,-8.00,22.00,970.00


In [30]:
duration_df[
    ["TheatreRoom", "ProcedureDescription", "ExpectedDurationMins", "operation_length_mins", "duration_error_mins"]
].sort_values("duration_error_mins", ascending=False).head(10)

,TheatreRoom,ProcedureDescription,ExpectedDurationMins,operation_length_mins,duration_error_mins
5582,IR LAB 3,Open embolectomy of cerebral artery,0.00,970.00,970.00
8103,BRUNEL TH 06,Unspecified opening of cranium,150.00,1017.00,867.00
9121,BRUNEL TH 07,Posterior fusion of joint of cervical spine NEC,180.00,971.00,791.00
8273,GYNAE TH 01,Elective lower uterine segment caesarean delivery,60.00,790.00,730.00
2317,BRUNEL TH 06,Other specified extirpation of lesion of meninges of brain,150.00,779.00,629.00
6969,BRUNEL TH 01,Debridement of burnt skin of head or neck,60.00,680.00,620.00
9522,BRUNEL TH 04,Shaft of tibia NEC,90.00,655.00,565.00
13037,BRUNEL TH 04,Left sided operation,60.00,586.00,526.00
13448,BRUNEL TH 09,Primary total prosthetic replacement of hip joint using cement,120.00,642.00,522.00
13504,BRUNEL TH 09,Primary total prosthetic replacement of hip joint using cement,120.00,642.00,522.00


## 5. Categorical variable inspection

In [31]:
categorical_columns = [
    "TheatreRoom",
    "admission_type",
    "SessionIDdesc",
    "actual_proc_1_procedure_code",
    "ProcedureDescription",
    "PriorityLevelCode",
    "anaesthetic_desc",
    "listing_cons_code",
    "theat_surg_1_national_code",
    "theat_anae_1_national_code",
]

pd.DataFrame({
    "unique_values": nbt_smallset_df[categorical_columns].nunique(dropna=True),
    "missing_count": nbt_smallset_df[categorical_columns].isna().sum(),
    "missing_pct": (nbt_smallset_df[categorical_columns].isna().mean() * 100).round(2),
}).sort_values("unique_values", ascending=False)

,unique_values,missing_count,missing_pct
SessionIDdesc,2288,0,0.00
actual_proc_1_procedure_code,1038,90,0.60
ProcedureDescription,1038,90,0.60
theat_surg_1_national_code,390,115,0.77
theat_anae_1_national_code,323,5406,36.24
listing_cons_code,283,77,0.52
TheatreRoom,45,0,0.00
admission_type,16,2,0.01
anaesthetic_desc,7,744,4.99
PriorityLevelCode,4,8563,57.40


In [32]:
for column in ["TheatreRoom", "ProcedureDescription", "PriorityLevelCode", "anaesthetic_desc", "ASAScore"]:
    print(f"\nTop values for {column}")
    display(nbt_smallset_df[column].value_counts(dropna=False).head(10).to_frame("count"))


Top values for TheatreRoom


,count
TheatreRoom,
BRUNEL TH 05,706
PLASTIC MINOR 1,651
IR LAB 1,631
GYNAE TH 02,621
IR LAB 2,551
CATH LAB 05,524
PLASTIC MINOR 2,523
BRUNEL TH 03,520
BRUNEL TH 06,516



Top values for ProcedureDescription


,count
ProcedureDescription,
Unspecified other excision of lesion of skin,1423
Elective lower uterine segment caesarean delivery,633
Other specified general anaesthetic,326
Coronary arteriography NEC,278
Lower uterine segment caesarean delivery NEC,259
Maintenance of drainage tube of kidney,223
Debridement of skin NEC,215
Total cholecystectomy NEC,181
Percutaneous transluminal angioplasty of artery,178



Top values for PriorityLevelCode


,count
PriorityLevelCode,
NaN,8563
P2,3714
P3,1521
P4,1119
P1,1



Top values for anaesthetic_desc


,count
anaesthetic_desc,
GA,8192
LA (Surgeon),4113
NaN,744
Neuraxial Block,655
No Anaesthetic,460
Sedation,307
GA and Regional,307
Regional Only,140



Top values for ASAScore


,count
ASAScore,
2.00,6219
3.00,4241
1.00,2424
NaN,1254
4.00,723
5.00,35
6.00,21
0.00,1


## 6. Time column inspection

These columns look like time-of-day or duration values. Before modeling or cleaning, check whether they parse consistently.

In [33]:
time_columns = [
    "into_theatre",
    "anaesthetic_start_time",
    "incision",
    "closure",
    "out_of_theatre",
    "operation_end_time",
    "recovery_time",
]

time_parse_summary = []
for column in time_columns:
    parsed = pd.to_timedelta(nbt_smallset_df[column].astype("string"), errors="coerce")
    time_parse_summary.append({
        "column": column,
        "non_null_original": nbt_smallset_df[column].notna().sum(),
        "parsed_successfully": parsed.notna().sum(),
        "parse_failed_non_null": (nbt_smallset_df[column].notna() & parsed.isna()).sum(),
        "zero_times": (parsed == pd.Timedelta(0)).sum(),
    })

pd.DataFrame(time_parse_summary)

,column,non_null_original,parsed_successfully,parse_failed_non_null,zero_times
0,into_theatre,14918,14918,0,612
1,anaesthetic_start_time,14918,14918,0,612
2,incision,14862,14862,0,539
3,closure,14845,14845,0,557
4,out_of_theatre,14321,14321,0,3
5,operation_end_time,14877,14877,0,540
6,recovery_time,1768,1768,0,37


In [34]:
time_as_delta = nbt_smallset_df[time_columns].apply(lambda series: pd.to_timedelta(series.astype("string"), errors="coerce"))

time_sequence_checks = pd.DataFrame({
    "anaesthetic_before_into_theatre": (time_as_delta["anaesthetic_start_time"] < time_as_delta["into_theatre"]).sum(),
    "incision_before_anaesthetic": (time_as_delta["incision"] < time_as_delta["anaesthetic_start_time"]).sum(),
    "closure_before_incision": (time_as_delta["closure"] < time_as_delta["incision"]).sum(),
    "out_before_closure": (time_as_delta["out_of_theatre"] < time_as_delta["closure"]).sum(),
}, index=["row_count"]).T

time_sequence_checks

,row_count
anaesthetic_before_into_theatre,0
incision_before_anaesthetic,6547
closure_before_incision,5597
out_before_closure,52


## 7. Useful relationship checks

In [35]:
nbt_smallset_df.groupby("anaesthetic_desc", dropna=False).agg(
    cases=("operation_length_mins", "size"),
    median_operation_length=("operation_length_mins", "median"),
    mean_operation_length=("operation_length_mins", "mean"),
    median_expected_duration=("ExpectedDurationMins", "median"),
).sort_values("cases", ascending=False)

,cases,median_operation_length,mean_operation_length,median_expected_duration
anaesthetic_desc,,,,
GA,8192,102.00,129.82,90.00
LA (Surgeon),4113,37.00,46.81,60.00
NaN,744,73.00,83.90,60.00
Neuraxial Block,655,73.00,79.68,60.00
No Anaesthetic,460,35.00,39.84,45.00
GA and Regional,307,118.00,132.36,60.00
Sedation,307,46.50,57.33,60.00
Regional Only,140,71.00,88.42,60.00


In [36]:
nbt_smallset_df.groupby("TheatreRoom").agg(
    cases=("operation_length_mins", "size"),
    median_operation_length=("operation_length_mins", "median"),
    mean_operation_length=("operation_length_mins", "mean"),
    missing_operation_length=("operation_length_mins", lambda series: series.isna().sum()),
).sort_values("cases", ascending=False).head(15)

,cases,median_operation_length,mean_operation_length,missing_operation_length
TheatreRoom,,,,
BRUNEL TH 05,706,81.00,101.14,6
PLASTIC MINOR 1,651,31.00,35.31,1
IR LAB 1,631,54.00,65.75,3
GYNAE TH 02,621,72.00,76.02,7
IR LAB 2,551,52.00,62.79,5
CATH LAB 05,524,61.00,69.18,3
PLASTIC MINOR 2,523,28.00,34.24,2
BRUNEL TH 03,520,120.50,135.99,4
BRUNEL TH 06,516,116.00,137.03,2


In [ ]:
nbt_smallset_df.groupby("ProcedureDescription", dropna=False).agg(
    cases=("operation_length_mins", "size"),
    median_operation_length=("operation_length_mins", "median"),
    mean_operation_length=("operation_length_mins", "mean"),
).query("cases >= 20").sort_values("median_operation_length", ascending=False).head(20)

## 8. Professional visual inspection

Use these plots to understand completeness, distributions, category balance, duration errors, and relationships between important variables.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

plot_color = "#2F6F9F"
highlight_color = "#D55E00"
neutral_color = "#6C757D"

### 8.1 Missing values

This shows which columns need the most attention before cleaning or modeling.

In [ ]:
missing_pct = (nbt_smallset_df.isna().mean() * 100).sort_values(ascending=True)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(10, 7))
missing_pct.plot(kind="barh", ax=ax, color=highlight_color)
ax.set_title("Missing values by column")
ax.set_xlabel("Missing values (%)")
ax.set_ylabel("")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
ax.set_xlim(0, min(100, missing_pct.max() + 12))

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(nbt_smallset_df.isna(), aspect="auto", interpolation="nearest", cmap="Greys")
ax.set_title("Missing value pattern across rows")
ax.set_xlabel("Columns")
ax.set_ylabel("Rows")
ax.set_xticks(range(len(nbt_smallset_df.columns)))
ax.set_xticklabels(nbt_smallset_df.columns, rotation=90)

plt.tight_layout()

### 8.2 Numeric distributions

These plots show the shape of important numeric columns and help reveal skew, outliers, and impossible values.

In [ ]:
plot_columns = ["ExpectedDurationMins", "operation_length_mins", "age_at_operation", "ASAScore"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.ravel()

for ax, column in zip(axes, plot_columns):
    nbt_smallset_df[column].dropna().plot(kind="hist", bins=40, ax=ax, color=plot_color, edgecolor="white")
    ax.set_title(f"Distribution of {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Row count")

plt.tight_layout()

In [ ]:
duration_columns = ["ExpectedDurationMins", "operation_length_mins"]

fig, ax = plt.subplots(figsize=(9, 5))
nbt_smallset_df[duration_columns].plot(kind="box", ax=ax)
ax.set_title("Outlier check for planned and actual duration")
ax.set_ylabel("Minutes")

plt.tight_layout()

### 8.3 Categorical distributions

These plots show the most common rooms, procedures, priority codes, anaesthetic types, and ASA scores.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.ravel()

category_plots = {
    "Top theatre rooms": nbt_smallset_df["TheatreRoom"].value_counts().head(15),
    "Top procedures": nbt_smallset_df["ProcedureDescription"].value_counts().head(15),
    "Priority level counts": nbt_smallset_df["PriorityLevelCode"].fillna("Missing").value_counts(),
    "Anaesthetic type counts": nbt_smallset_df["anaesthetic_desc"].fillna("Missing").value_counts(),
}

for ax, (title, values) in zip(axes, category_plots.items()):
    values.sort_values().plot(kind="barh", ax=ax, color=plot_color)
    ax.set_title(title)
    ax.set_xlabel("Row count")
    ax.set_ylabel("")

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

nbt_smallset_df["ASAScore"].astype("string").fillna("Missing").value_counts().sort_index().plot(kind="bar", ax=axes[0], color=plot_color)
axes[0].set_title("ASA score counts")
axes[0].set_xlabel("ASA score")
axes[0].set_ylabel("Row count")

nbt_smallset_df["sex_national_code"].astype("string").fillna("Missing").value_counts().sort_index().plot(kind="bar", ax=axes[1], color=neutral_color)
axes[1].set_title("Sex national code counts")
axes[1].set_xlabel("Sex national code")
axes[1].set_ylabel("Row count")

plt.tight_layout()

### 8.4 Relationships and performance checks

These visuals compare planned duration with actual operation length and show where duration differs by room or anaesthetic type.

In [ ]:
duration_plot_df = nbt_smallset_df[["ExpectedDurationMins", "operation_length_mins"]].dropna()

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(
    duration_plot_df["ExpectedDurationMins"],
    duration_plot_df["operation_length_mins"],
    alpha=0.25,
    s=12,
    color=plot_color,
)

max_duration = duration_plot_df[["ExpectedDurationMins", "operation_length_mins"]].quantile(0.99).max()
ax.plot([0, max_duration], [0, max_duration], color=highlight_color, linestyle="--", label="planned = actual")
ax.set_xlim(0, max_duration)
ax.set_ylim(0, max_duration)
ax.set_title("Planned duration vs actual operation length")
ax.set_xlabel("Expected duration mins")
ax.set_ylabel("Operation length mins")
ax.legend()

plt.tight_layout()

In [ ]:
duration_error = nbt_smallset_df["operation_length_mins"] - nbt_smallset_df["ExpectedDurationMins"]
duration_error_trimmed = duration_error[duration_error.between(duration_error.quantile(0.01), duration_error.quantile(0.99))]

fig, ax = plt.subplots(figsize=(10, 5))
duration_error_trimmed.dropna().plot(kind="hist", bins=50, ax=ax, color=plot_color, edgecolor="white")
ax.axvline(0, color=highlight_color, linestyle="--", label="actual = planned")
ax.set_title("Distribution of duration error")
ax.set_xlabel("Operation length minus expected duration, minutes")
ax.set_ylabel("Row count")
ax.legend()

plt.tight_layout()

In [ ]:
duration_by_anaesthetic = (
    nbt_smallset_df.assign(anaesthetic_desc=nbt_smallset_df["anaesthetic_desc"].fillna("Missing"))
    .groupby("anaesthetic_desc")["operation_length_mins"]
    .median()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 5))
duration_by_anaesthetic.plot(kind="barh", ax=ax, color=plot_color)
ax.set_title("Median operation length by anaesthetic type")
ax.set_xlabel("Median operation length, minutes")
ax.set_ylabel("")

plt.tight_layout()

In [ ]:
top_rooms = nbt_smallset_df["TheatreRoom"].value_counts().head(12).index
room_duration_df = nbt_smallset_df[nbt_smallset_df["TheatreRoom"].isin(top_rooms)]

fig, ax = plt.subplots(figsize=(14, 6))
room_duration_df.boxplot(column="operation_length_mins", by="TheatreRoom", ax=ax, rot=75)
ax.set_title("Operation length distribution for busiest theatre rooms")
ax.set_xlabel("Theatre room")
ax.set_ylabel("Operation length, minutes")
fig.suptitle("")

plt.tight_layout()

## 9. Initial findings to investigate

- The dataset has 14,918 rows and 24 columns.
- `recovery_time` is mostly missing, so it may not be reliable unless recovery data is optional or captured elsewhere.
- `PriorityLevelCode` and `theat_anae_1_national_code` also have high missingness.
- There are a few exact duplicate rows that should be reviewed before modeling.
- Some duration fields include zeros and very large values, so outlier checks are important.
- Time columns are stored as objects, so they should be converted carefully before time-based feature engineering.
- `theatre_notes` can contain useful information, but it is free text and should be cleaned separately from structured columns.

## Recommended next steps

1. Decide the project target: prediction, cleaning/reporting, delay analysis, theatre utilization, or case-duration modeling.
2. Create a cleaned copy of the dataframe instead of overwriting the raw dataframe.
3. Standardize column names to snake_case.
4. Convert categorical code columns to string/category types.
5. Convert time columns using `pd.to_timedelta` and document how overnight cases should be handled.
6. Decide how to handle missing values column by column.
7. Review duplicate rows and impossible/suspicious duration values before removing anything.

## References

NHS Data Dictionary (2019) *Admission Method*. Available at: https://archive.datadictionary.nhs.uk/DD%20Release%20May%202019/data_dictionary/attributes/a/add/admission_method_de.asp%40shownav%3D1.html (Accessed: 25 July 2026).

NHS Data Dictionary (2024) *Intended Management*. Available at: https://archive.datadictionary.nhs.uk/DD%20Release%20May%202024/attributes/intended_management.html (Accessed: 25 July 2026).

NHS Data Dictionary (2024) *ASA Physical Status Classification System Code*. Available at: https://archive.datadictionary.nhs.uk/DD%20Release%20May%202024/attributes/asa_physical_status_classification_system_code.html (Accessed: 25 July 2026).

NHS England (n.d.) *National Elective Access Policy*. Available at: https://www.england.nhs.uk/long-read/national-elective-access-policy/ (Accessed: 25 July 2026).

NHS England Digital (n.d.) *Clinical classifications*. Available at: https://digital.nhs.uk/services/terminology-and-classifications/clinical-classifications (Accessed: 25 July 2026).
